In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')


folder_name = "Ctrl+Alt+Del_DataVers_AnavaUGM"
base_path = f'/content/drive/MyDrive/{folder_name}'


if os.path.exists(base_path):
    print(f"Berhasil terhubung ke folder: {folder_name}")
    print("Isi folder:", os.listdir(base_path))
else:
    print(f"Folder tidak ditemukan! Pastikan nama folder di Drive sesuai: {folder_name}")

data_path = os.path.join(base_path, 'data')
notebooks_path = os.path.join(base_path, 'notebooks')
models_path = os.path.join(base_path, 'models')
sub_path = os.path.join(base_path, 'submissions')

# Contoh cara pakai nanti:
# df = pd.read_csv(f'{data_path}/raw/train.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Berhasil terhubung ke folder: Ctrl+Alt+Del_DataVers_AnavaUGM
Isi folder: ['notebooks', 'data', 'models', 'submissions']


In [ ]:
#load data
import pandas as pd
import numpy as np

data_train = pd.read_csv(f'{data_path}/train.csv')
data_test = pd.read_csv(f'{data_path}/test.csv')
df_train = pd.DataFrame(data_train)
df_test = pd.DataFrame(data_test)

In [ ]:
#simpan trip id untuk submission
test_ids = df_test["Trip_ID"].copy()

# drop timestamp dan trip id
df_train = df_train.drop(columns=['Trip_ID', 'Timestamp'])
df_test  = df_test.drop(columns=['Timestamp'])

In [ ]:
# cek missing value
df_train.info()
df_test.info()

missing_value = df_train.isna().sum()
missing_percentage = (missing_value / len(df_train)) * 100

df_missing_value = pd.DataFrame({"Missing Value":missing_value, "Missing Percentage":missing_percentage})
df_missing_value

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000000 entries, 0 to 7999999
Data columns (total 23 columns):
 #   Column            Dtype  
---  ------            -----  
 0   Pickup_Lat        float64
 1   Pickup_Long       float64
 2   Dropoff_Lat       float64
 3   Dropoff_Long      float64
 4   GPS_Accuracy_M    float64
 5   Distance_KM       float64
 6   Est_Price_IDR     float64
 7   Surge_Multiplier  float64
 8   Accel_X           float64
 9   Accel_Y           float64
 10  Accel_Z           float64
 11  Gyro_Z            float64
 12  Pickup_Zone       object 
 13  Dropoff_Zone      object 
 14  Device_FP         object 
 15  Promo_Code        object 
 16  Car_Model         object 
 17  Payment_Method    object 
 18  Weather           object 
 19  Traffic           object 
 20  Battery_Level     object 
 21  Signal_Strength   object 
 22  Trip_Label        object 
dtypes: float64(12), object(11)
memory usage: 1.4+ GB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000000 

,Missing Value,Missing Percentage
Pickup_Lat,929348,11.616850
Pickup_Long,611083,7.638538
Dropoff_Lat,1914440,23.930500
Dropoff_Long,1556029,19.450363
GPS_Accuracy_M,1504090,18.801125
Distance_KM,941317,11.766462
Est_Price_IDR,1151401,14.392512
Surge_Multiplier,612307,7.653838
Accel_X,1608442,20.105525
Accel_Y,1081337,13.516713


In [ ]:
def clean_battery(df):
    df["Battery_Level"] = (
        df["Battery_Level"]
        .astype(str)
        .str.replace("%", "", regex=False)
        .replace("nan", np.nan)
        .astype(float)
    )
    return df

df_train = clean_battery(df_train)
df_test  = clean_battery(df_test)

In [ ]:
cat_cols_category = ["Pickup_Zone", "Dropoff_Zone", "Car_Model"]
cat_cols_str = ["Device_FP", "Promo_Code", "Payment_Method","Weather", "Traffic", "Signal_Strength"]

#ubah object ke category dan diisi unknown (train)
for col in cat_cols_category:
    df_train[col] = df_train[col].astype("category")
    if "Unknown" not in df_train[col].cat.categories:
        df_train[col] = df_train[col].cat.add_categories(["Unknown"])
    df_train[col] = df_train[col].fillna("Unknown")

#ubah object ke str dan diisi unknown (train)
for col in cat_cols_str:
    df_train[col] = df_train[col].astype(str).fillna("Unknown")

#ubah object ke category dan diisi unknown (test)
for col in cat_cols_category:
    df_test[col] = df_test[col].astype("category")
    df_test[col] = df_test[col].cat.set_categories(
        df_train[col].cat.categories
    )
    df_test[col] = df_test[col].fillna("Unknown")

#ubah object ke str dan diisi unknown (test)
for col in cat_cols_str:
    df_test[col] = df_test[col].astype(str).fillna("Unknown")

#ubah float64 ke float32
float_cols = df_train.select_dtypes(include="float64").columns
df_train[float_cols] = df_train[float_cols].astype(np.float32)
df_test[float_cols]  = df_test[float_cols].astype(np.float32)

print(df_train.info())
print(df_test.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000000 entries, 0 to 7999999
Data columns (total 23 columns):
 #   Column            Dtype   
---  ------            -----   
 0   Pickup_Lat        float32 
 1   Pickup_Long       float32 
 2   Dropoff_Lat       float32 
 3   Dropoff_Long      float32 
 4   GPS_Accuracy_M    float32 
 5   Distance_KM       float32 
 6   Est_Price_IDR     float32 
 7   Surge_Multiplier  float32 
 8   Accel_X           float32 
 9   Accel_Y           float32 
 10  Accel_Z           float32 
 11  Gyro_Z            float32 
 12  Pickup_Zone       category
 13  Dropoff_Zone      category
 14  Device_FP         object  
 15  Promo_Code        object  
 16  Car_Model         category
 17  Payment_Method    object  
 18  Weather           object  
 19  Traffic           object  
 20  Battery_Level     float32 
 21  Signal_Strength   object  
 22  Trip_Label        object  
dtypes: category(3), float32(13), object(7)
memory usage: 863.4+ MB
None
<class 'pandas

In [ ]:
#pake 2 juta data aja
df_train_sample = df_train.sample(frac=0.25, random_state=42)

In [ ]:
# pisah x y
x = df_train_sample.drop(columns='Trip_Label')
y = df_train_sample['Trip_Label']

In [ ]:
!pip install catboost

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

from catboost import CatBoostClassifier

In [ ]:
# split
x_train, x_val, y_train, y_val = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=8,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    random_seed=42,
    verbose=100,
    bootstrap_type="Bernoulli",
    subsample=0.5
)

In [ ]:
model.fit(
    x_train,
    y_train,
    cat_features=cat_cols_category + cat_cols_str,
    eval_set=(x_val, y_val),
    use_best_model=True
)

0:	learn: 0.6986879	test: 0.6987878	best: 0.6987878 (0)	total: 31s	remaining: 2h 34m 30s


In [ ]:
#evaluasi
y_val_pred = model.predict(x_val)
print("Macro F1:", f1_score(y_val, y_val_pred, average="macro"))
print(classification_report(y_val, y_val_pred))

In [ ]:
#prediksi
x_test = df_test.drop(columns="Trip_ID")
y_test_pred = model.predict(x_test)

In [ ]:
#file submission
submission = pd.DataFrame({
    "Trip_ID": test_ids,
    "Trip_Label": y_test_pred
})

submission.to_csv("submission.csv", index=False)